# RAG Playground: Modüler Bileşenlerle Asistan İnşası

## RAG Mimari Akışı

Doküman Yükleme -> Text Extraction -> Chunking -> Embedding -> VectorDB'ye Ekleme -> Sorgu / Guardrails -> Intent -> Retrieval (Dense + BM25 -> RRF -> Reranker) -> Cevap Üretimi (LLM-as-a-Judge) -> Hafıza Güncelleme

---

> [!TIP]
> **Yardıma mı ihtiyacınız var?**
> Yapboz parçalarını birleştirirken veya parametrelerin pratik etkilerini anlamakta yardıma ihtiyaç duyarsanız **`notebooks/demo.ipynb`** dosyasına bakabilirsiniz. `demo.ipynb`, her modülün canlı çalışan referans kodlarını, Manim görselleştirmelerini ve adım adım testlerini içerir.

---

## Adım Adım Kullanılacak Sınıflar, Import Yolları ve Fonksiyonlar

Aşağıda mimariyi oluşturan her bir yapboz parçasının nereden import edileceği, görevi, kurucu (`__init__`) parametreleri ve çağrılacak fonksiyonları listelenmiştir.

### 0. Temel LLM İstemcisi (`LLMFactory`)
Model API çağrılarını sağlayıcıdan (Azure OpenAI, OpenAI, Gemini) bağımsızlaştıran temel istemci fabrikasıdır.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.core.settings import settings
  from rag_playground.llm.llm_factory import LLMFactory
  ```
- **Parametreler (`create` metodu)**:
  - `provider` (`str`, varsayılan `settings.LLM_PROVIDER`): Kullanılacak LLM sağlayıcısı (`azure`, `openai`, `gemini`).
  - `timeout` (`int`, varsayılan `60`): Model API istekleri için zaman aşımı süresi (saniye).
  - `max_retries` (`int`, varsayılan `3`): Ağ hatası veya rate limit durumunda yeniden deneme sayısı.
- **Çağrılacak Fonksiyon / Kullanım**:
  - `client = LLMFactory.create(provider=settings.LLM_PROVIDER)`
  - *Açıklama:* Bu metot, diğer tüm servislere (`embedder`, `retriever`, `guardrails`, `intent_classifier`, `memory_service`, `assistant`) aktarılacak temel `LLMClient` örneğini üretir.

### 1. Doküman Girişi ve Metin Çıkarma (`DocumentIO` & `DocumentExtractor`)
Ham PDF dokümanını okur, kaydeder ve yapılandırılmış Markdown metnine dönüştürür.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.ingestion.document_io import DocumentIO
  from rag_playground.ingestion.extractor import DocumentExtractor
  ```
- **Parametreler**:
  - **`DocumentIO`**:
    - `encoding` (`str`, varsayılan `'utf-8'`): Metin dosyaları için karakter kodlaması.
  - **`DocumentExtractor`**:
    - `ocr_enabled` (`bool`, varsayılan `False`): Taranmış belgeler için OCR motorunu açar.
    - `force_backend_text` (`bool`, varsayılan `False`): Doğrudan backend metin çıkarımını zorlar.
    - `image_scale` (`float`, varsayılan `3.0`): Sayfa render görüntü ölçeği.
    - `ocr_factory` (`OCRFactory | None`, varsayılan `None`): OCR dil ve sayfa ayarları fabrikası.
    - `layout_enabled` (`bool`, varsayılan `True`): Başlık ve sayfa düzeni analizi yapar.
    - `table_enabled` (`bool`, varsayılan `True`): Tabloları satır/sütun yapısıyla Markdown'a çevirir.
    - `do_cell_matching` (`bool`, varsayılan `True`): Tablo hücre koordinat eşleştirmesini aktif eder.
    - `do_formula_enrichment` (`bool`, varsayılan `True`): Matematiksel formülleri LaTeX gösterimine çevirir.
    - `generate_picture_images` (`bool`, varsayılan `True`): Görsel ve tablo resimlerini ayıklar.
- **Çağrılacak Fonksiyonlar**:
  - `content = extractor.extract(file_path=saved_pdf_path)`
    - PDF dosyasını okuyup başlıkları, tabloları ve paragrafları düzenlenmiş bir Markdown metni (`str`) olarak çıkarır.
  - `saved_path = await document_io.save_document(content=content, filename=filename, save_dir=save_dir, file_type=FileType.MD)`
    - Yüklenen PDF dosyasını veya çıkarılan Markdown metnini diske asenkron olarak kaydeder.
  - `text, hash_value = document_io.read_file(file_path)`
    - Diskteki bir metin dosyasını ve SHA-256 özetini okur.

### 2. Akıllı Parçalama (`DocumentChunker`)
Metni başlık hiyerarşisi (`breadcrumb`), tablolar ve cümle embedding mesafelerine göre anlamsal parçalara böler.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.ingestion.chunker import DocumentChunker
  ```
- **Parametreler**:
  - `embedder` (`EmbeddingService`, zorunlu): Cümleler arası anlamsal mesafeyi hesaplayan servis.
  - `max_chunk_chars` (`int`, varsayılan `700`): Parçanın karakter üst hedefi.
  - `min_chunk_chars` (`int`, varsayılan `260`): Küçük parçaların öncekiyle birleştirilme alt sınırı.
  - `sentence_overlap_ratio` (`float`, varsayılan `0.2`): Önceki parçadan taşınacak cümle örtüşme oranı (%20).
  - `semantic_chunking_break_percentage` (`float`, varsayılan `0.85`): Semantik kopma persentil eşiği.
  - `semantic_chunking_min_dist` (`float`, varsayılan `0.2`): Bölme için gereken minimum kosinüs mesafesi ($1 - \text{CosineSimilarity}$).
  - `semantic_chunking_buffer_size` (`int`, varsayılan `1`): Cümle vektörü için komşu penceresi (önceki + mevcut + sonraki).
  - `semantic_chunking_min_sentences` (`int`, varsayılan `5`): Semantik bölme uygulanacak asgari cümle sayısı.
  - `process_logging` (`bool`, varsayılan `False`): Parçalama loglarını konsola yazdırır.
- **Çağrılacak Fonksiyon**:
  - `chunks = await chunker.chunk(content=content, document_path=saved_pdf_path)`
    - Ayıklanan metni alır ve hiyerarşik/semantik olarak `list[Citation]` listesine böler. Her bir parça; metin içeriği, sayfa numarası, paragraf numarası ve ait olduğu başlık hiyerarşisini barındırır.

### 3. Vektörleştirme (`LLMEmbeddingService`)
Metinleri çok boyutlu embedding vektörlerine dönüştürür.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.retrieval.embedding_service import LLMEmbeddingService
  ```
- **Parametreler**:
  - `client` (`LLMClient`, zorunlu): Embedding API çağrısını yapacak istemci.
  - `deployment_name` (`str`, zorunlu): Kullanılacak embedding model adı (`settings.llm_models.embedding`).
  - `embed_batch_size` (`int`, varsayılan `96`): Tek istekte gönderilecek maksimum metin sayısı.
  - `timeout` (`int`, varsayılan `60`): Zaman aşımı süresi (saniye).
- **Çağrılacak Fonksiyon**:
  - `vectors = await embedder.embed_texts(texts=["örnek metin 1", "örnek metin 2"])`
    - Verilen metin listesini embedding vektörlerine (`list[list[float]]`) dönüştürür.
    - *Not:* Parçaları veritabanına kaydederken ve ararken vektörleştirme adımı çoğu zaman `AsyncQdrantVectorDB` ve `RetrievalService` tarafından arka planda otomatik yürütülür.

### 4. Vektör Veritabanı ve Depolama (`AsyncQdrantVectorDB`)
Metinleri ve vektörleri Qdrant koleksiyonuna kaydeder; eşzamanlı olarak bellek içi BM25 indeksini yönetir.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.retrieval.vector_db import AsyncQdrantVectorDB
  ```
- **Parametreler**:
  - `embedder` (`EmbeddingService`, zorunlu): Arama ve kayıt sırasında metinleri vektöre dönüştürecek servis.
  - `mode` (`str`, varsayılan `settings.QDRANT_MODE`): Qdrant calisma modu (`local`, `memory` veya `remote`).
  - `path` (`str`, varsayılan `settings.QDRANT_PATH`): Local Qdrant verilerinin saklanacagı dizin.
  - `api_key` (`str | None`, varsayılan `None`): Vektör veritabanı erişim anahtarı.
  - `bm25_k1` (`float`, varsayılan `1.5`): BM25 terim sıklığı doygunluk katsayısı.
  - `bm25_b` (`float`, varsayılan `0.75`): BM25 doküman uzunluğu normalizasyon katsayısı.
  - `upsert_batch_size` (`int`, varsayılan `128`): Qdrant'a tek seferde gönderilecek kayıt sayısı.
- **Çağrılacak Fonksiyonlar**:
  - `await vector_db.store(records=vector_records, collection_name="Documents", vectors=vectors)`
    - `VectorRecord` listesini ve embedding vektörlerini Qdrant koleksiyonuna ve eşzamanlı olarak bellek içi BM25 indeksine yazar.
  - `hits = await vector_db.search(query=query, collection_name="Documents", limit=5, candidate_k=10)`
    - Hem Dense (vektör) hem de Sparse (BM25) aramayı birleştirerek hibrit arama sonuçlarını RRF (Reciprocal Rank Fusion) ile harmanlayıp döndürür (`list[SearchHit]`).
  - `hits = await vector_db.dense_search(query=query, limit=5, collection_name="Documents")`
    - Yalnızca anlamsal (kosinüs benzerliği) vektör araması yapar.

### 5. Hibrit Arama ve Yeniden Sıralama (`RetrievalService`)
Dense (vektör) ve Sparse (BM25) aramayı RRF ile birleştirir; ardından adayları LLM hakemiyle puanlayarak filtreler.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.retrieval.retrieval_service import RetrievalService
  ```
- **Parametreler**:
  - `vector_db` (`VectorDB`, zorunlu): Hibrit aramanın yapılacağı veritabanı.
  - `client` (`LLMClient`, zorunlu): Puanlama yapacak model istemcisi.
  - `reranker_active` (`bool`, varsayılan `True`): LLM ile yeniden sıralamayı açar/kapatır.
  - `reranker` (`str`, varsayılan `settings.llm_models.reranker`): Puanlayıcı deployment adı (`gpt-4o-mini`).
  - `reranker_system_prompt` (`str`, varsayılan `RERANK_SYSTEM_PROMPT`): Hakemin puanlama kriterleri.
  - `temperature` (`float`, varsayılan `0.0`): Puanlama tutarlılığı için sıcaklık.
  - `rerank_passage_chars` (`int`, varsayılan `700`): LLM'e gönderilecek maksimum pasaj karakteri.
  - `min_rerank_score` (`int`, varsayılan `3`): Bu puanın altındaki parçalar elenir (0-10 arası).
  - `response_format` (`dict`, varsayılan `LLM_RERANK_RESPONSE_FORMAT`): Yapılandırılmış JSON şeması.
  - `timeout` (`int`, varsayılan `30`): Zaman aşımı süresi (saniye).
- **Çağrılacak Fonksiyonlar**:
  - `count = await retriever.index_citations(citations=chunks, collection_name="Documents")`
    - `DocumentChunker`'dan dönen `chunks` (`list[Citation]`) listesini doğrudan alır, `VectorRecord` nesnelerine dönüştürür ve tek adımda vektör veritabanına indeksler (en pratik kayıt yöntemi!).
  - `citations = await retriever.retrieve(query=query, top_k=5)`
    - Hibrit arama (Dense + BM25) yapar, RRF ile birleştirir, reranker devredeyse LLM hakemiyle puanlayıp eşiği geçen en alakalı `list[Citation]` parçalarını döndürür.

### 6. Güvenlik ve Çıktı Denetimi (`GuardrailsService`)
Girdide Prompt Injection engeller, PII (TCKN, IBAN, Kredi Kartı, Telefon) maskeler; çıktıda LLM-as-a-Judge ile halüsinasyon denetimi ve geri bildirim (feedback) döngüsü sağlar.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.assistant.guardrails import GuardrailsService
  ```
- **Parametreler**:
  - `mode` (`Literal["strict", "lenient", "loose"]`, varsayılan `"strict"`): Maskeleme kapsamı katılığı.
  - `regex_patterns` (`list[Pattern] | None`, varsayılan `None`): Özel regex maskeleme kuralları.
  - `proprietary_dictionary` (`list[str] | None`, varsayılan `None`): Kuruma özel gizli sözcükler.
  - `normalization_form` (`Literal["NFC", "NFD", "NFKC", "NFKD"]`, varsayılan `"NFKC"`): Unicode normalizasyonu.
  - `llm_as_judge_enabled` (`bool`, varsayılan `True`): Çıktı denetimini açar/kapatır.
  - `client` (`LLMClient | None`, varsayılan `None`): Hakem denetimi model istemcisi.
  - `temperature` (`float | None`, varsayılan `0.0`): Hakem sıcaklık değeri.
  - `guardrails_deployment` (`str | None`, varsayılan `settings.llm_models.guardrails`): Hakem model adı (`gpt-4o-mini`).
  - `llm_as_judge_prompt` (`str | None`, varsayılan `LLM_AS_JUDGE_PROMPT`): Hakem talimatı.
  - `response_format` (`dict | None`, varsayılan `LLM_GUARDRAILS_FORMAT`): JSON şeması.
  - `timeout` (`int | None`, varsayılan `30`): Zaman aşımı süresi.
- **Çağrılacak Fonksiyonlar**:
  - `is_blocked, sanitized_text = guardrails.input_guardrail(text=user_query)`
    - Girdide Prompt Injection olup olmadığını denetler (engellenirse `is_blocked=True`) ve PII verilerini maskeler.
  - `sanitized_answer = guardrails.output_guardrail(answer=model_answer)`
    - Model çıktısındaki PII veya hassas terimleri maskeler.
  - `is_blocked, feedback, sanitized_answer = await guardrails.llm_as_judge(question=q, answer=a, citations=citations, memory=memories)`
    - Model cevabını kaynaklara ve soruya göre LLM hakemiyle denetler; halüsinasyon veya güvenlik ihlali varsa geri bildirim (`feedback`) üretir.
  - *Not:* `AssistantService`, `guardrails_enabled=True` iken bu metotları arka planda otomatik yürütür.

### 7. Niyet Sınıflandırma ve Sorgu Çözümleme (`IntentClassifier`)
Kullanıcı mesajının niyetini belirler (`knowledge_question`, `chitchat`, `needs_clarification`, `out_of_scope`) ve zamirleri çözümlenmiş bağımsız arama sorgusunu (`resolved_query`) üretir.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.assistant.intent import IntentClassifier
  ```
- **Parametreler**:
  - `client` (`LLMClient`, zorunlu): Sınıflandırma yapacak LLM istemcisi.
  - `system_prompt` (`str`, varsayılan `INTENT_SYSTEM_PROMPT`): Niyet sınıflandırma talimatı.
  - `classifier_model` (`str`, varsayılan `settings.llm_models.intent`): Model deployment adı (`gpt-4o-mini`).
  - `temperature` (`float`, varsayılan `0.0`): Sıcaklık.
  - `response_format` (`dict`, varsayılan `LLM_INTENT_FORMAT`): JSON çıktı şeması.
  - `timeout` (`int`, varsayılan `30`): Zaman aşımı süresi.
- **Çağrılacak Fonksiyon**:
  - `intent = await intent_classifier.classify_intent(query=user_query, history=chat_history)`
    - Kullanıcı sorusunu ve önceki konuşma bağlamını inceleyerek niyet nesnesini (`Intent`) döndürür. `intent.intent` niyet kategorisini, `intent.resolved_query` ise zamirlerin (örn. "o nedir?" -> "RAG mimarisi nedir?") açılmış halini verir.
  - *Not:* `AssistantService` bu fonksiyonu her gelen kullanıcı sorusunda otomatik çağırır.

### 8. Konuşma ve Kullanıcı Belleği (`MemoryService`)
Kısa süreli konuşma geçmişini (Rolling Window) ve uzun süreli kullanıcı tercihlerini/olgularını (Semantic Memory) yönetir.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.memory.memory_service import MemoryService
  ```
- **Parametreler**:
  - `client` (`LLMClient`, zorunlu): Hafıza çıkarımı yapacak istemci.
  - `vector_db` (`VectorDB`, zorunlu): Anıların saklanacağı veritabanı.
  - `user_id` (`str`, zorunlu): Kullanıcı kimliği (ayrım için).
  - `history_max_messages` (`int`, varsayılan `6`): Kısa süreli bellekte tutulacak maksimum mesaj çifti.
  - `history_max_tokens` (`int`, varsayılan `2000`): Kısa süreli bellek maksimum token sınırı.
  - `long_term_memory_batch_size` (`int`, varsayılan `10`): Uzun süreli aktarımı tetikleyen mesaj sayısı.
  - `long_term_memory_min_score` (`float`, varsayılan `0.5`): Anı eşleşmesi için gereken benzerlik skoru.
  - `memory_extraction_min_confidence` (`float`, varsayılan `0.7`): Bilginin hafızaya yazılması için asgari güven eşiği.
  - `memory_extraction_model` (`str`, varsayılan `settings.llm_models.memory`): Bellek çıkarım modeli (`gpt-4o-mini`).
  - `system_prompt` (`str`, varsayılan `MEMORY_SYSTEM_PROMPT`): Bellek çıkarım talimatı.
  - `memory_response_format` (`dict`, varsayılan `LLM_MEMORY_RESPONSE_FORMAT`): JSON şeması.
  - `timeout` (`int`, varsayılan `30`): Zaman aşımı süresi.
- **Çağrılacak Fonksiyonlar**:
  - `memory_service.update_short_term_memory(role="user", content=user_query)`
    - Kısa süreli konuşma hafızasına yeni kullanıcı veya asistan mesajı ekler.
  - `history = memory_service.get_short_term_memories()`
    - Asistanın bağlama eklemesi için son konuşma mesajlarını (`list[dict]`) getirir.
  - `memories = await memory_service.get_long_term_memories(query=query, limit=3)`
    - Kullanıcının önceki oturumlarından kaydedilmiş kalıcı olguları/tercihleri vektörel aramayla getirir.
  - `memory_service.reset_history()`
    - Kısa vadeli geçmişi sıfırlar ve biriken konuşmaları uzun süreli belleğe aktarır.
  - *Not:* `AssistantService` (`memory_enabled=True`), konuşma akışında bu fonksiyonları otomatik yürütür.

### 9. Asistan Orkestrasyonu (`AssistantService`)
Tüm bu modülleri bir araya getiren ana orkestratördür.
- **Nereden Import Edilir**:
  ```python
  from rag_playground.assistant.assistant import AssistantService
  ```
- **Parametreler**:
  - `client` (`LLMClient`, zorunlu): Sohbet yanıtını üretecek model istemcisi.
  - `intent_classifier` (`IntentClassifier`, zorunlu): Niyet belirleyici servis.
  - `retriever` (`RetrievalService`, zorunlu): Kaynak dokümanları getiren servis.
  - `guardrails` (`GuardrailsService | None`, varsayılan `None`): Güvenlik denetçisi.
  - `memory_service` (`MemoryIService | None`, varsayılan `None`): Bellek servisi.
  - `guardrails_enabled` (`bool`, varsayılan `True`): Guardrails ve LLM-as-a-Judge'ı açar/kapatır.
  - `memory_enabled` (`bool`, varsayılan `True`): Bellek entegrasyonunu açar/kapatır.
  - `model` (`str`, varsayılan `settings.llm_models.chat`): Yanıtı üretecek ana sohbet modeli (`gpt-5.4` / `gpt-4o`).
  - `temperature` (`float`, varsayılan `0.3`): Üretim çeşitliliği (RAG için 0.2 - 0.4 önerilir).
  - `system_prompt` (`str`, varsayılan `ASSISTANT_SYSTEM_PROMPT`): Asistanın davranış ve kaynak kuralları.
  - `timeout` (`int`, varsayılan `30`): Zaman aşımı süresi.
- **Çağrılacak Fonksiyonlar**:
  - `response = await chatbot.answer(user_input=user_query, reset_history=False)`
    - Asistanın ana giriş kapısıdır. Guardrails -> Intent -> Retrieval -> Memory -> Answer Generation -> LLM-as-a-Judge adımlarını baştan sona otomatik koşturur.
    - Dönüş: `AssistantResponse` nesnesi (`response.answer`, `response.citations`, `response.intent`).
  - `print(AssistantService.show_citations(response.citations))`
    - Yanıtta kullanılan kaynak dokümanların başlıklarını, sayfa ve paragraf numaralarını, rerank puanlarını şık ve okunabilir bir metin olarak biçimlendirir.


In [ ]:
# ==============================================================================
# DOKÜMAN YÜKLEME WIDGET'I
#
# Nereden Import Edilir:
# from utils import create_upload_widget, save_uploaded_file
#
# Girdi Parametreleri (Inputs):
# - create_upload_widget(): Parametre almaz, PDF seçme widget'ını oluşturup ekranda görüntüler.
# - save_uploaded_file(upload_widget, save_dir=None, default_filename="AS19_Sayi-RAG-Sistemlerine-Giris.pdf"):
#     * upload_widget: create_upload_widget() tarafından üretilen FileUpload nesnesi.
#     * save_dir (Path, opsiyonel): PDF'in kaydedileceği klasör (Varsayılan: data/documents/raw_documents).
#     * default_filename (str, opsiyonel): Dosya yüklenmezse kullanılacak varsayılan dosya adı.
#
# Nasıl Kullanılır:
# 1. Bu hücreyi çalıştırın ve 'Upload' butonundan bilgisayarınızdan bir PDF seçin.
# 2. 'saved_pdf_path' değişkeni seçilen veya varsayılan PDF'in dosya yolunu alır.
# ==============================================================================

import importlib
import utils
importlib.reload(utils)
from utils import create_upload_widget, save_uploaded_file

upload = create_upload_widget()
saved_pdf_path = await save_uploaded_file(upload)


In [ ]:
# ==============================================================================
# ÖĞRENCİ ÇALIŞMA ALANI: KENDİ ASİSTANINIZI İNŞA EDİN (TUVAL)
#
# Yukarıdaki parametre, import ve fonksiyon rehberine bakarak yapboz parçalarını burada birleştirin.
# Takıldığınız her an 'notebooks/demo.ipynb' dosyasına bakabilirsiniz!
#
# Gerekli Importlar:
# from rag_playground.core.settings import settings
# from rag_playground.llm.llm_factory import LLMFactory
# from rag_playground.ingestion.extractor import DocumentExtractor
# from rag_playground.ingestion.chunker import DocumentChunker
# from rag_playground.retrieval.embedding_service import LLMEmbeddingService
# from rag_playground.retrieval.vector_db import AsyncQdrantVectorDB
# from rag_playground.retrieval.retrieval_service import RetrievalService
# from rag_playground.assistant.guardrails import GuardrailsService
# from rag_playground.assistant.intent import IntentClassifier
# from rag_playground.memory.memory_service import MemoryService
# from rag_playground.assistant.assistant import AssistantService
#
# Adım 1: LLMFactory.create(provider=settings.LLM_PROVIDER) ile 'client' nesnenizi oluşturun.
# Adım 2: DocumentExtractor ve DocumentChunker nesnelerini kurun.
# Adım 3: extractor.extract(file_path=saved_pdf_path) ile metni çıkarın ve chunks = await chunker.chunk(...) ile parçalayın.
# Adım 4: LLMEmbeddingService ve AsyncQdrantVectorDB nesnelerini oluşturup parçaları kaydedin (await retriever.index_citations(chunks)).
# Adım 5: RetrievalService, GuardrailsService, IntentClassifier ve MemoryService nesnelerini tanımlayın.
# Adım 6: Tüm bu parçaları AssistantService altında 'chatbot' adıyla birleştirin.
# Adım 7: response = await chatbot.answer("...") ile test edin veya yukarıdaki interaktif Chat Widget'ında deneyin!
# ==============================================================================

# Kendi kodlarınızı bu alana yazabilirsiniz:





In [ ]:
# ==============================================================================
# İNTERAKTİF KONUŞMA KUTUCUĞU (CHAT WIDGET'I)
#
# Nereden Import Edilir:
# from utils import create_chat_widget
#
# Girdi Parametreleri (Inputs):
# - chatbot (AssistantService | None): Aşağıdaki çalışma alanında (tuval) oluşturacağınız
#     asistan nesnesi. Eğer verilmezse çalışma alanındaki global 'chatbot' değişkenini otomatik arar.
#
# Nasıl Kullanılır:
# 1. Bir sonraki çalışma alanında (tuval) asistanınızı 'chatbot' adıyla oluşturup o hücreyi çalıştırın.
# 2. Ardından bu hücreyi çalıştırarak konuşma arayüzünü açın ve sorularınızı sorun.
# 3. İsterseniz demo.ipynb'deki gibi doğrudan 'await ask("sorunuz", chatbot=chatbot)' fonksiyonunu da kullanabilirsiniz!
# ==============================================================================

import importlib
import utils
importlib.reload(utils)
from utils import create_chat_widget

create_chat_widget(chatbot=globals().get("chatbot"))
